## Natural language processing

After using CTS to download the full text of speeches, you might want to perform more sophisticated, language-specific analysis. At this point, some users will be confident applying natural language processing tools with which they are familiar. Meanwhile, the DICES client itself has some basic integration with the spaCy framework for NLP. Using the `nlp_spacy` extension, you can call spaCy to parse the downloaded text of a speech and return a set of **tokens** (words and punctuation marks) that are tagged for dictionary headword, part of speech, and morphological roles such as case, mood, etc.

**Note**: Language models for NLP are a form of artificial intelligence and necessarily involve a certain amount of uncertainty and error. Forms may be parsed incorrectly, and indeed the same form may be parsed differently (rightly or wrongly) in different contexts. It's important to test your results and decide what level of accuracy you feel comfortable with.

### Using spaCy

[spaCy](https://spacy.io/) is a framework for running NLP tasks that is model-independent. Models are provided by third parties for various languages and spaCy provides a common interface and workflow. For Latin, we ourselves use the [LatinCy](https://spacy.io/universe/project/latincy) family of models, and for Ancient Greek, [OdyCy](https://spacy.io/universe/project/odycy), but there are alternatives you can explore as well.

Some considerations: larger models are more accurate, but use more memory and compute. Personal computers or cloud-based platforms with limited resources may need to use the "small" model in a given family.

### Installing models

The models that we use are packaged as "wheels" for installation into a Python environment using `pip` the Python package manager. They live on HuggingFace, a hosting platform for machine learning, where they usually have a small description or "model card" and a collection of files. You can find the wheel file and either download it manually to your machine or copy the download URL to provide to `pip`.

#### Example: LatinCy

The model card for the "small" LatinCy model, called `la_core_web_sm`, is at https://huggingface.co/latincy/la_core_web_sm. It has information about the version number, authors of the model, and accuracy scores. If I click on "Files and versions", I can see the set of files associated with the model, including the wheel file, which ends in `.whl`. The interface will let me download the file directly, or copy the download link, https://huggingface.co/latincy/la_core_web_sm/resolve/main/la_core_web_sm-3.9.4-py3-none-any.whl.

#### Method 1: install from `pip`

The simplest way to install the model in my system is to point `pip` directly to the download URL. At the command line, I use
```
pip install https://huggingface.co/latincy/la_core_web_sm/resolve/main/la_core_web_sm-3.9.4-py3-none-any.whl
```
Or in Jupyter,
```
!pip install https://huggingface.co/latincy/la_core_web_sm/resolve/main/la_core_web_sm-3.9.4-py3-none-any.whl
```

#### Method 2: download and install from local copy

Alternatively, I could click on the download button in my browser and save the wheel file to my computer. Then from the command line I would do something like the following (adjust for wherever the file lands when you download it, and add the leading `!` if you're in Jupyter):
```
pip install /Users/chris/Downloads/la_core_web_sm-3.9.4-py3-none-any.whl
```

#### **Special Note**: some models don't play nicely with pip!

Some models from 2024, especially the small Greek models, have wheel files that follow an older convention and no longer work with modern `pip`. The fix is a small one, though:
 - The wheel file name should include a version number right after the model name
 - Older wheels may have the word `any` instead
 - You need to download the file, change the `any` to a version-like number, and then `pip install` the renamed wheel

**Example**

The [small OdyCy model](https://huggingface.co/chcaa/grc_odycy_joint_sm) (but not the big one) has this problem. [On HuggingFace](https://huggingface.co/chcaa/grc_odycy_joint_sm/tree/main) it's called `grc_odycy_joint_sm-any-py3-none-any.whl`. The first `any` in its name doesn't agree with `pip`'s naming conventions and so it can't be installed from the URL. To install, I have to
 - download the file
 - rename `grc_odycy_joint_sm-any-py3-none-any.whl` to e.g. `grc_odycy_joint_sm-0.0.1-py3-none-any.whl` (the actual version number doesn't matter.)
 - install the renamed file with pip, e.g., `pip install grc_odycy_joint_sm-0.0.1-py3-none-any.whl`


### Tldr;

😰 Phew, that sounds like a lot! It's true that NLP is still a dynamic field and can be a little tricky to get running on any particular computer. On the other hand, often all it takes is some patience. The steps below work for us on most computers, including Google Colab's free tier, and hopefully provide a place to get started.

## NLP with DICES client

### Install prerequisites

To use the DICES client's built-in wrappers for spaCy, it's necessary to install a few Python packages and the language models. The three lines below work for us, installing the largest models for both Greek and Latin.

**Note on Jupyter**: Because we're in a Jupyter notebook, Python is *already running* and hasn't had a chance to recognize and initialize the models. After running the cell below, **you must restart the kernel** before the rest of the notebook will work. (From the JupyterLab menu, select "Kernel" -> "Restart kernel...". After you restart the kernel, you don't need to re-run this install process; just carry on with later cells.

**Note on Colab**: In Google Colab, you may be prompted to restart the kernel automatically after this step. Otherwise, choose "Runtime" -> "Restart Session" and then carry on with the cells below. For Colab only, you'll have to redo the model installation process every time you re-open your notebook, because the environment is not persistent.

In [ ]:
!pip install 'dices-client[spacy] @ git+https://github.com/cwf2/dices-client.git'
!pip install 'https://huggingface.co/latincy/la_core_web_trf/resolve/main/la_core_web_trf-3.9.4-py3-none-any.whl'
!pip install 'https://huggingface.co/chcaa/grc_odycy_joint_trf/resolve/main/grc_odycy_joint_trf-0.7.0-py3-none-any.whl'

**One final note:** You will likely see some error messages about spacy version conflicts here. This is unavoidable when we're combining Greek and Latin models with different requirements, but (for us at least) it doesn't cause any fatal crashes in the code.

### After installing the models...

Okay,
 - you've installed the models
 - you've restarted the kernel

Now, carry on below.

### Initialize DICES, CTS, and NLP

Create a new DICES session as usual, initalize CTS in order to work with texts (see `Ex_03 - Text.ipynb`), and also initialize NLP. The `initializeNlp()` call requires that you specify the names of the models you want to work with. You must give at least one of either `latin_model` or `greek_model`.

In [1]:
from dicesapi import DicesAPI

# initialize connection to DICES
api = DicesAPI()

# enable text download
api.initializeCts()

# enable parsing with NLP
api.initializeNlp(
    latin_model = "la_core_web_trf",
    greek_model = "grc_odycy_joint_trf",
)

/Users/chris/Documents/git/dices-project-container/dices-examples/venv/lib/python3.11/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'la_core_web_trf' (3.9.4) was trained with spaCy v3.8.11 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


### Query DICES for speeches

Let's start with all the speeches by Achilles, not only in the *Iliad* but across our corpus.

In [2]:
# query the DICES database for a set of speeches
speeches = api.getSpeeches(spkr_name='Achilles')

# use Counter to tally speeches by work
from collections import Counter

dist = Counter([speech.work.title for speech in speeches])

print(dist) 

Counter({'Iliad': 88, 'Posthomerica': 8, 'Achilleid': 7, 'Metamorphoses': 5, 'Odyssey': 3})


#### Select a test set

Because NLP can take time and resources, we'll work on just a subset of these speeches: let's select the first 5 in Greek and the first 5 in Latin.

In [3]:
test_set = speeches.filterBy("lang", "greek")[:5] + speeches.filterBy("lang", "latin")[:5]

# sort the results (using the + operator can shuffle the contents of SpeechGroups)
test_set.sort()

for speech in test_set:
    print(speech)

<Speech 4: Iliad 1.59-1.67>
<Speech 6: Iliad 1.85-1.91>
<Speech 9: Iliad 1.122-1.129>
<Speech 11: Iliad 1.149-1.171>
<Speech 13: Iliad 1.202-1.205>
<Speech 2269: Metamorphoses 12.80-12.81>
<Speech 2271: Metamorphoses 12.106-12.114>
<Speech 2272: Metamorphoses 12.120-12.121>
<Speech 2274: Metamorphoses 12.177-12.181>
<Speech 2303: Metamorphoses 13.445-13.448>


### Download the text from Perseus

See `Ex_03 - Text.ipynb` for more details.

In [4]:
for speech in test_set:
    speech.fetchPassage()

print(len([speech for speech in test_set if speech.passage]), "succeeded")

10 succeeded


### Parse with spaCy

To process a given speech with spaCy, use the `runSpacyPipeline()` method.
 - NLP has to be run for each speech you want to parse
 - `fetchPassage()` must already have been run, and `.passage` must exist on the speech
 - now run `passage.runSpacyPipeline()` to parse the passage text.

spaCy's NLP pipeline produces a spaCy-specific object called a [Doc](https://spacy.io/api/doc), which for our purposes we treat as a container of tokens. Our `runSpacyPipeline()` method will determine the model to use based on the speech's `lang` attribute, run the pipeline, and store the resulting Doc as `passage.spacy_doc`.

In [5]:
for speech in test_set:
    speech.passage.runSpacyPipeline()

print(len([speech for speech in test_set if speech.passage.spacy_doc]), "succeeded")

/Users/chris/Documents/git/dices-project-container/dices-examples/venv/lib/python3.11/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/Users/chris/Documents/git/dices-project-container/dices-examples/venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


10 succeeded


## Exploring results

Let's explore some of the information we can retrieved from the parsed tokens.

In [6]:
# select just the first speech
speech = test_set[0]

# if you dump the parsed speech to screen, by default it prints the text
print(speech.passage.spacy_doc)
print()

# count words
print(len(speech.passage.spacy_doc), "tokens")
print()

# count off the first 10 tokens
for i, tok in enumerate(speech.passage.spacy_doc[:10]):
    print(i, tok)

Ἀτρεΐδη νῦν ἄμμε παλιμπλαγχθέντας ὀΐω ἂψ ἀπονοστήσειν, εἴ κεν θάνατόν γε φύγοιμεν, εἰ δὴ ὁμοῦ πόλεμός τε δαμᾷ καὶ λοιμὸς Ἀχαιούς· ἀλλʼ ἄγε δή τινα μάντιν ἐρείομεν ἢ ἱερῆα ἢ καὶ ὀνειροπόλον, καὶ γάρ τʼ ὄναρ ἐκ Διός ἐστιν, ὅς κʼ εἴποι ὅ τι τόσσον ἐχώσατο Φοῖβος Ἀπόλλων, εἴτʼ ἄρʼ ὅ γʼ εὐχωλῆς ἐπιμέμφεται ἠδʼ ἑκατόμβης, αἴ κέν πως ἀρνῶν κνίσης αἰγῶν τε τελείων βούλεται ἀντιάσας ἡμῖν ἀπὸ λοιγὸν ἀμῦναι.

78 tokens

0 Ἀτρεΐδη
1 νῦν
2 ἄμμε
3 παλιμπλαγχθέντας
4 ὀΐω
5 ἂψ
6 ἀπονοστήσειν
7 ,
8 εἴ
9 κεν


Unlike Python's native `split()` method, the OdyCy model recognized the comma as separate from the word it followed, even though there was no whitespace.

Each token has its own attributes, including dictionary headword and part of speech, assigned by the model.

In [7]:
# count off the first 10 tokens
for i, tok in enumerate(speech.passage.spacy_doc[:10]):
    print(i, tok, tok.pos_)

0 Ἀτρεΐδη NOUN
1 νῦν ADV
2 ἄμμε PRON
3 παλιμπλαγχθέντας VERB
4 ὀΐω VERB
5 ἂψ ADV
6 ἀπονοστήσειν VERB
7 , PUNCT
8 εἴ SCONJ
9 κεν PART


### Convert tokens to a table

I think it's easier to examine a series of tokens in tabular form. Let's use Pandas to make a DataFrame.

In [8]:
import pandas as pd

# select just the first speech
speech = test_set[0]

# start with an empty list
rows = []

# iterate over tokens
for token in speech.passage.spacy_doc:

    # create a record for just this token
    row = {
        "token": token.text,       # original inflected form
        "lemma": token.lemma_,     # the dictionary headword
        "pos":   token.pos_,       # part of speech tag
        "morph": token.morph,      # morphological features
    }
    # add to list
    rows.append(row)

# now convert list of dicts to pandas df
tokens = pd.DataFrame(rows)

# display first 20 rows
tokens.head(20)

,token,lemma,pos,morph
0,Ἀτρεΐδη,ἀτρείδης,NOUN,"(Case=Voc, Gender=Masc, Number=Sing)"
1,νῦν,νῦν,ADV,()
2,ἄμμε,ἐγώ,PRON,"(Case=Acc, Gender=Masc, Number=Plur)"
3,παλιμπλαγχθέντας,παλιμπλάζομαι,VERB,"(Case=Acc, Gender=Masc, Number=Plur, Tense=Pas..."
4,ὀΐω,οἴομαι,VERB,"(Mood=Ind, Number=Sing, Person=1, Tense=Pres, ..."
5,ἂψ,ἄψ,ADV,()
6,ἀπονοστήσειν,ἀπονοστέω,VERB,"(Tense=Fut, VerbForm=Inf, Voice=Act)"
7,",",",",PUNCT,()
8,εἴ,εἰ,SCONJ,()
9,κεν,ἄν,PART,()


#### Notes on spaCy annotation standards

**Part of speech** tags used by spaCy models follow the [Universal POS Tag schema](https://universaldependencies.org/u/pos/).
**Morphological features** are drawn from the [Universal Features](https://universaldependencies.org/u/feat/index.html) set.

### Breaking out `morph` features

Morphological features are stored as a set by default, and spaCy has its own tools and idioms for working with them. I personally find it easier to break them out as strings and put them into the table. I'm including my recipe below. Note that not only are all the morphological features bundled as a set, but also each feature can take a list of values (though most don't in practice in Greek and Latin). Making each feature a column like this means that for most tokens, many columns are empty: nouns, for example, have no tense.

In [9]:
# select just the first speech
speech = test_set[0]

# start with an empty list
rows = []

# iterate over tokens
for token in speech.passage.spacy_doc:

    # create a record for just this token
    row = {
        "token": token.text,       # original inflected form
        "lemma": token.lemma_,     # the dictionary headword
        "pos":   token.pos_,       # part of speech tag
        "verbform": ";".join(token.morph.get("VerbForm")),   # finite, infinitive, ppl, etc.
        "mood":     ";".join(token.morph.get("Mood")),       # mood
        "tense":    ";".join(token.morph.get("Tense")),      # tense
        "voice":    ";".join(token.morph.get("Voice")),      # voice
        "person":   ";".join(token.morph.get("Person")),     # person
        "number":   ";".join(token.morph.get("Number")),     # number
        "case":     ";".join(token.morph.get("Case")),       # case
        "gender":   ";".join(token.morph.get("Gender")),     # gender
    }
    # add to list
    rows.append(row)

# now convert list of dicts to pandas df
tokens = pd.DataFrame(rows)

# display first 10 rows
tokens.head(10)

,token,lemma,pos,verbform,mood,tense,voice,person,number,case,gender
0,Ἀτρεΐδη,ἀτρείδης,NOUN,,,,,,Sing,Voc,Masc
1,νῦν,νῦν,ADV,,,,,,,,
2,ἄμμε,ἐγώ,PRON,,,,,,Plur,Acc,Masc
3,παλιμπλαγχθέντας,παλιμπλάζομαι,VERB,Part,,Past,Pass,,Plur,Acc,Masc
4,ὀΐω,οἴομαι,VERB,Fin,Ind,Pres,Act,1,Sing,,
5,ἂψ,ἄψ,ADV,,,,,,,,
6,ἀπονοστήσειν,ἀπονοστέω,VERB,Inf,,Fut,Act,,,,
7,",",",",PUNCT,,,,,,,,
8,εἴ,εἰ,SCONJ,,,,,,,,
9,κεν,ἄν,PART,,,,,,,,


### Latin example

Now that we've shown that the recipe works for Greek, let's look at a Latin example.

In [10]:
# select just the first speech
speech = test_set.filterBy("lang", "latin")[0]

# start with an empty list
rows = []

# iterate over tokens
for token in speech.passage.spacy_doc:

    # create a record for just this token
    row = {
        "token": token.text,       # original inflected form
        "lemma": token.lemma_,     # the dictionary headword
        "pos":   token.pos_,       # part of speech tag
        "verbform": ";".join(token.morph.get("VerbForm")),   # finite, infinitive, ppl, etc.
        "mood":     ";".join(token.morph.get("Mood")),       # mood
        "tense":    ";".join(token.morph.get("Tense")),      # tense
        "voice":    ";".join(token.morph.get("Voice")),      # voice
        "person":   ";".join(token.morph.get("Person")),     # person
        "number":   ";".join(token.morph.get("Number")),     # number
        "case":     ";".join(token.morph.get("Case")),       # case
        "gender":   ";".join(token.morph.get("Gender")),     # gender

    }
    # add to list
    rows.append(row)

# now convert list of dicts to pandas df
tokens = pd.DataFrame(rows)

# display first 25 rows
tokens.head(25)

,token,lemma,pos,verbform,mood,tense,voice,person,number,case,gender
0,“,“,PUNCT,,,,,,,,
1,quisquis,quisquis,DET,,,,,,Sing,Nom,Masc
2,es,sum,AUX,Fin,Ind,Pres,,2,Sing,,
3,",",",",PUNCT,,,,,,,,
4,o,o,INTJ,,,,,,,,
5,iuuenis,iuuenis,NOUN,,,,,,Sing,Nom,Masc
6,”,”,PUNCT,,,,,,,,
7,dixit,dico,VERB,Fin,Ind,Past,Act,3,Sing,,
8,“,“,PUNCT,,,,,,,,
9,solamen,solamen,NOUN,,,,,,Sing,Acc,Neut


## Export results as a single table

At this point you might want to export the parsed tokens to Excel. Here we convert the full test set to a single table and export in CSV format using Pandas.

In [11]:
# start with an empty list
rows = []

# iterate over all speeches
for speech in test_set:

    # iterate over tokens in this speech
    for token in speech.passage.spacy_doc:

        # create a record
        row = {
            # speech-level attributes
            "speech": speech.id,
            "author": speech.work.author.name,
            "work": speech.work.title,
            "speaker": speech.getSpkrString(), # this function concatenates all speakers as a single string
            "addressee": speech.getAddrString(), # likewise for addressees

            # token-level attributes
            "token": token.text,       # original inflected form
            "lemma": token.lemma_,     # the dictionary headword
            "pos":   token.pos_,       # part of speech tag
            "verbform": ";".join(token.morph.get("VerbForm")),   # finite, infinitive, ppl, etc.
            "mood":     ";".join(token.morph.get("Mood")),       # mood
            "tense":    ";".join(token.morph.get("Tense")),      # tense
            "voice":    ";".join(token.morph.get("Voice")),      # voice
            "person":   ";".join(token.morph.get("Person")),     # person
            "number":   ";".join(token.morph.get("Number")),     # number
            "case":     ";".join(token.morph.get("Case")),       # case
            "gender":   ";".join(token.morph.get("Gender")),     # gender
        }
        
        # add to list
        rows.append(row)
    
# now convert list of dicts to pandas df
tokens = pd.DataFrame(rows)

# export to file
tokens.to_csv("token_export.csv", index=False)